In [11]:
import numpy as np
import xarray as xr
from ndsl.logging import ndsl_log
import datetime as dt
from pace import NullComm
from ndsl import CubedSphereCommunicator, SubtileGridSizer, GridIndexing, QuantityFactory, Backend

In [49]:
def read_o3_data(
    ntoz: int, o3_data: str
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Reads ozone data from a NetCDF file and returns it as an xarray DataArray.

    Args:
        ntoz (int): The number of ozone tracer species
        o3_data (string): The path to the NetCDF file containing ozone data.
    """
    if ntoz <= 0:
        raise NotImplementedError("Diagnostic ozone (ntoz <= 0) is not supported")

    ds = xr.open_dataset(o3_data)
    time = ds['time'].values
    oz_coeff = ds['ozcoeff'].values
    oz_pres = np.log(100 * ds['lev'].values)
    levozp = len(ds['lev'].values)
    ozlat = ds['lat'].values
    ozplin = ds['ozplin'].values
    ds.close()

    ndsl_log.info(f"Read ozone data from {o3_data}")
    ndsl_log.info(f"oz_coeff: {(oz_coeff)}")
    ndsl_log.info(f"latsozp: {len(ozlat)}")
    ndsl_log.info(f"levozp: {levozp}")
    ndsl_log.info(f"timeoz: {len(time)}")

    return levozp, oz_coeff, ozlat, oz_pres, time, ozplin

def setindexoz(dlat: np.ndarray, oz_lat: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    jindx1 = np.zeros(dlat.shape, dtype=int)
    jindx2 = np.zeros(dlat.shape, dtype=int)
    ddy = np.zeros(dlat.shape, dtype=float)
    for j, k in np.ndindex(dlat.shape):
        jindx2[j, k] = len(oz_lat)
        for i in range(len(oz_lat)):
            if dlat[j, k] < oz_lat[i]:
                jindx2[j, k] = i
                break
        jindx1[j, k] = max(jindx2[j, k]-1, 0)
        jindx2[j, k] = min(jindx2[j, k], len(oz_lat))
        if jindx2[j, k] != jindx1[j, k]:
            ddy[j, k] = (dlat[j, k] - oz_lat[jindx1[j, k]]) / (
                oz_lat[jindx2[j, k]] - oz_lat[jindx1[j, k]]
            )
        else:
            ddy[j, k] = 1.0
    return jindx1, jindx2, ddy


def ozinterpolate(
    model_time: dt.datetime,
    jindx1: np.ndarray,
    jindx2: np.ndarray,
    ddy: np.ndarray,
    oz_time: np.ndarray,
    oz_coeff: int,
    levozp: int,
    ozplin: np.ndarray
) -> np.ndarray:
    """
    Interpolate ozone onto model levels at a given time in-place onto ozplout
    """
    jdoy = int(model_time.strftime("%j"))  # day of year 1-365
    rjday = jdoy + (model_time.hour / 24.)
    if rjday < oz_time[0]:
        rjday += 365.

    n2 = len(oz_time)
    for j in range(1, len(oz_time) - 1):
        if rjday < oz_time[j]:
            n2 = j
            break
    n1 = n2 - 1
    tx1 = (oz_time[n2] - rjday) / (oz_time[n2] - oz_time[n1])
    tx2 = 1.0 - tx1
    ozplout = np.zeros((ddy.shape[0], ddy.shape[1], levozp, oz_coeff))
    for nc in range(oz_coeff):
        for ll in range(levozp):
            for j, k in np.ndindex(ddy.shape):
                j1 = jindx1[j, k]
                j2 = jindx2[j, k]
                tem = 1.0 - ddy[j, k]
                ozplout[j, k, ll, nc] = tx1 * (
                    tem * ozplin[n1, nc, ll, j1] + ddy[j, k] * ozplin[n1, nc, ll, j2]
                ) + tx2 * (
                    tem * ozplin[n2, nc, ll, j1] + ddy[j, k] * ozplin[n2, nc, ll, j2]
                )
    return ozplout


In [12]:
comm = NullComm(0,6,-1.0)
communicator = CubedSphereCommunicator.from_layout(comm, (1,1))

backend = Backend("st:numpy:cpu:IJK")

sizer = SubtileGridSizer.from_tile_params(
    nx_tile=24,
    ny_tile=24,
    nz=79,
    n_halo=3,
    layout=(1,1),
    tile_partitioner=communicator.partitioner.tile,
    tile_rank=communicator.tile.rank,
    backend=backend,
)

grid_indexing = GridIndexing.from_sizer_and_communicator(
    sizer=sizer, comm=communicator
)
quantity_factory = QuantityFactory(
    sizer, backend=backend
)

In [13]:
from ndsl.grid import (
    AngleGridData,
    ContravariantGridData,
    DampingCoefficients,
    DriverGridData,
    GridData,
    HorizontalGridData,
    MetricTerms,
    VerticalGridData,
)
eta_file = "eta79.nc"

metric_terms = MetricTerms(
    quantity_factory=quantity_factory,
    communicator=communicator,
    eta_file=eta_file,
)
horizontal_data = HorizontalGridData.new_from_metric_terms(metric_terms)
vertical_data = VerticalGridData.new_from_metric_terms(metric_terms)
contravariant_data = ContravariantGridData.new_from_metric_terms(metric_terms)
angle_data = AngleGridData.new_from_metric_terms(metric_terms)
grid_data = GridData(
    horizontal_data=horizontal_data,
    vertical_data=vertical_data,
    contravariant_data=contravariant_data,
    angle_data=angle_data,
)

damping_coefficients = DampingCoefficients.new_from_metric_terms(metric_terms)
driver_grid_data = DriverGridData.new_from_metric_terms(metric_terms)

/usr/local/lib/python3.12/site-packages/ndsl/grid/generation.py:1549: RuntimeWarning: divide by zero encountered in divide
  data=1.0 / self.area[:],
/usr/local/lib/python3.12/site-packages/ndsl/grid/gnomonic.py:681: RuntimeWarning: invalid value encountered in divide
  np.sum(p * q, axis=-1) / np.sqrt(np.sum(p**2, axis=-1) * np.sum(q**2, axis=-1))
/usr/local/lib/python3.12/site-packages/ndsl/grid/gnomonic.py:681: RuntimeWarning: invalid value encountered in scalar divide
  np.sum(p * q, axis=-1) / np.sqrt(np.sum(p**2, axis=-1) * np.sum(q**2, axis=-1))
/usr/local/lib/python3.12/site-packages/ndsl/grid/generation.py:1564: RuntimeWarning: divide by zero encountered in divide
  data=1.0 / self.area_c[:],
/usr/local/lib/python3.12/site-packages/ndsl/grid/gnomonic.py:177: RuntimeWarning: invalid value encountered in divide
  return (xyz.T / ((xyz**2).sum(axis=-1) ** 0.5).T).T
/usr/local/lib/python3.12/site-packages/ndsl/grid/gnomonic.py:697: RuntimeWarning: invalid value encountered in divi

In [3]:
levozp, oz_coeff, ozlat, oz_pres, oztime, ozplin = read_o3_data(1, "ozprd.nc")

2026-05-29 14:53:59|INFO|rank 0|ndsl.logging:Read ozone data from ozprd.nc
2026-05-29 14:53:59|INFO|rank 0|ndsl.logging:oz_coeff: [1 2 3 4 5 6]
2026-05-29 14:53:59|INFO|rank 0|ndsl.logging:latsozp: 71
2026-05-29 14:53:59|INFO|rank 0|ndsl.logging:levozp: 53
2026-05-29 14:53:59|INFO|rank 0|ndsl.logging:timeoz: 13


In [4]:
levozp

53

In [16]:
dlat = grid_data.lat.field[:]
oz_lat = ozlat
jindx1 = np.zeros(dlat.shape, dtype=int)
jindx2 = np.zeros(dlat.shape, dtype=int)
ddy = np.zeros(dlat.shape, dtype=float)
for j, k in np.ndindex(dlat.shape):
    jindx2[j, k] = len(oz_lat)
    for i in range(len(oz_lat)):
        if dlat[j, k] < oz_lat[i]:
            jindx2[j, k] = i
            break
    jindx1[j, k] = max(jindx2[j, k]-1, 0)
    jindx2[j, k] = min(jindx2[j, k], len(oz_lat))
    if jindx2[j, k] != jindx1[j, k]:
        ddy[j, k] = (dlat[j, k] - oz_lat[jindx1[j, k]]) / (
            oz_lat[jindx2[j, k]] - oz_lat[jindx1[j, k]]
        )
    else:
        ddy[j, k] = 1.0

In [19]:
jindx1, jindx2, ddy = setindexoz(grid_data.lat.field[:], ozlat)

In [25]:
jindx2.shape

(25, 25)

In [50]:
prdout = ozinterpolate(
    dt.datetime.fromisoformat("2000-03-20T00:00:30Z"),
    jindx1,
    jindx2,
    ddy,
    oztime,
    len(oz_coeff),
    levozp,
    ozplin,
)

In [52]:
prdout.shape

(25, 25, 53, 6)

In [46]:
levozp

53

In [47]:
len(ozlat)

71

In [48]:
len(oztime)

13